# 05a - Content Representation: Model Selection

## Objective

Before integrating ASR, we first select a suitable text representation for
semantic content.

For a transcription $W$, we consider:

$$
Z_{\mathrm{content}}(W)=E(W).
$$

The goal is **not** to assume that `all-MiniLM-L6-v2` is the correct model.
Instead, we compare several sentence-embedding models on a small controlled
benchmark designed around the needs of this project:

1. **Paraphrases** should have high cosine similarity.
2. **Contradictions / negations** should be substantially less similar than
   paraphrases.
3. **Related but different content** should show some semantic similarity,
   while remaining lower than paraphrases when appropriate.
4. **Unrelated sentences** should have low similarity.
5. **Lexically overlapping but semantically different sentences** should not
   be considered highly similar simply because they share many words.

This notebook is only about the **text encoder**. ASR is handled separately
in `11_content_representation_asr.ipynb`.

The selected model will then be used to define:

$$
Z_{\mathrm{content}}(X)
=
E\left(\mathrm{ASR}(X)\right).
$$

In [38]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [39]:
benchmark = {
    "paraphrase": [
        (
            "I am studying statistics at EPFL.",
            "I study statistics at EPFL."
        ),
        (
            "I went to the supermarket yesterday.",
            "Yesterday, I bought some groceries."
        ),
        (
            "The meeting was postponed until Monday.",
            "The meeting has been moved to Monday."
        ),
    ],

    "contradiction": [
        (
            "I am studying statistics at EPFL.",
            "I do not study statistics at EPFL."
        ),
        (
            "The interview was recorded in Lausanne.",
            "The interview was not recorded in Lausanne."
        ),
        (
            "She works in finance.",
            "She does not work in finance."
        ),
        (
            "It rained the whole day.",
            "It did not rain during the day."
        ),
    ],

    "related": [
        (
            "The weather is very nice today.",
            "It rained the whole day."
        ),
        (
            "I am studying statistics at EPFL.",
            "I am taking a machine learning course at EPFL."
        ),
        (
            "I bought a new computer yesterday.",
            "My laptop stopped working this morning."
        ),
    ],

    "unrelated": [
        (
            "I am studying statistics at EPFL.",
            "The football match was cancelled because of the rain."
        ),
        (
            "My favorite programming language is Python.",
            "The restaurant closes at eleven tonight."
        ),
        (
            "I bought a new computer yesterday.",
            "The train arrived twenty minutes late."
        ),
    ],
        "lexical_overlap": [
        (
            "The bank is closed because it is Sunday.",
            "The bank is open because it is Sunday."
        ),
        (
            "I went to the bank to deposit some money.",
            "I sat on the bank of the river and watched the water."
        ),
        (
            "The crane lifted the heavy machine.",
            "The crane flew over the building."
        ),
        (
            "I saw a bat flying outside my house.",
            "He bought a new bat for the baseball game."
        ),
        (
            "The mouse is under the table.",
            "I bought a new mouse for my computer."
        ),
    ],
}

## 2. Candidate models

We keep the first comparison reasonably small so that it is feasible on a
local machine.

The models below represent different commonly used sentence-embedding
families. All of them are used through `SentenceTransformer`, and their
embeddings are L2-normalized before cosine similarity is computed.

We should **not** call one model "best" before running the benchmark.

In [40]:
MODELS = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "MPNet": "sentence-transformers/all-mpnet-base-v2",
    "BGE-small": "BAAI/bge-small-en-v1.5",
    "BGE-base": "BAAI/bge-base-en-v1.5",
}

## 3. Compute pairwise similarities

For each model, we encode the two sentences of every pair and compute:

$$
\mathrm{sim}(W_1,W_2)
=
\frac{E(W_1)^\top E(W_2)}
{\|E(W_1)\|\|E(W_2)\|}.
$$

Because the embeddings are normalized, this is simply their dot product.

In [41]:
def evaluate_model(model, benchmark):
    rows = []

    for category, pairs in benchmark.items():
        sentences = []
        for s1, s2 in pairs:
            sentences.extend([s1, s2])

        embeddings = model.encode(
            sentences,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        for i, (s1, s2) in enumerate(pairs):
            z1 = embeddings[2 * i]
            z2 = embeddings[2 * i + 1]
            similarity = float(np.dot(z1, z2))

            rows.append({
                "category": category,
                "sentence_1": s1,
                "sentence_2": s2,
                "cosine": similarity,
            })

    return pd.DataFrame(rows)

In [42]:
all_results = []

for model_label, model_name in MODELS.items():
    print("=" * 80)
    print(f"Loading: {model_label}")
    print(f"Model:  {model_name}")

    model = SentenceTransformer(
    model_name,
    device="cpu",
    )

    embedding_dim = model.get_embedding_dimension()
    print(f"Embedding dimension: {embedding_dim}")

    results = evaluate_model(model, benchmark)
    results["model"] = model_label
    results["embedding_dim"] = embedding_dim
    all_results.append(results)

    del model

results_df = pd.concat(all_results, ignore_index=True)

print("\nFinished.")
print("Number of evaluated pairs:", len(results_df))

Loading: MiniLM
Model:  sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 5127.21it/s]


Embedding dimension: 384
Loading: MPNet
Model:  sentence-transformers/all-mpnet-base-v2


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5940.48it/s]


Embedding dimension: 768
Loading: BGE-small
Model:  BAAI/bge-small-en-v1.5


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5792.63it/s]


Embedding dimension: 384
Loading: BGE-base
Model:  BAAI/bge-base-en-v1.5


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 4958.48it/s]


Embedding dimension: 768

Finished.
Number of evaluated pairs: 72


## Pair-level results

The complete pair-level results are retained so that we can inspect which examples drive the model differences.

In [43]:
display(
    results_df[
        ["model", "category", "cosine", "sentence_1", "sentence_2"]
    ].sort_values(["model", "category"])
)

,model,category,cosine,sentence_1,sentence_2
57,BGE-base,contradiction,0.746460,I am studying statistics at EPFL.,I do not study statistics at EPFL.
58,BGE-base,contradiction,0.845838,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
59,BGE-base,contradiction,0.798639,She works in finance.,She does not work in finance.
60,BGE-base,contradiction,0.790257,It rained the whole day.,It did not rain during the day.
67,BGE-base,lexical_overlap,0.895588,The bank is closed because it is Sunday.,The bank is open because it is Sunday.
...,...,...,...,...,...
8,MiniLM,related,0.616614,I am studying statistics at EPFL.,I am taking a machine learning course at EPFL.
9,MiniLM,related,0.507271,I bought a new computer yesterday.,My laptop stopped working this morning.
10,MiniLM,unrelated,0.060312,I am studying statistics at EPFL.,The football match was cancelled because of the rain.
11,MiniLM,unrelated,-0.088658,My favorite programming language is Python.,The restaurant closes at eleven tonight.


## Summary statistics

For every model and category we report the mean, standard deviation, minimum and maximum cosine similarity.

In [44]:
summary = (
    results_df
    .groupby(["model", "category"])["cosine"]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

display(summary)

mean    std    min    max
model     category                                   
BGE-base  contradiction    0.795  0.041  0.746  0.846
          lexical_overlap  0.704  0.129  0.583  0.896
          paraphrase       0.896  0.071  0.832  0.973
          related          0.731  0.069  0.673  0.807
          unrelated        0.399  0.049  0.344  0.439
BGE-small contradiction    0.814  0.044  0.754  0.851
          lexical_overlap  0.737  0.110  0.631  0.904
          paraphrase       0.917  0.060  0.869  0.985
          related          0.764  0.066  0.725  0.840
          unrelated        0.442  0.099  0.345  0.543
MPNet     contradiction    0.606  0.091  0.479  0.692
          lexical_overlap  0.518  0.142  0.405  0.740
          paraphrase       0.787  0.080  0.709  0.869
          related          0.418  0.185  0.212  0.573
          unrelated        0.025  0.093 -0.072  0.114
MiniLM    contradiction    0.769  0.043  0.740  0.833
          lexical_overlap  0.599  0.193  0.460  0.910
          paraphrase       0.799  0.098  0.713  0.906
          related          0.520  0.091  0.436  0.617
          unrelated        0.009  0.085 -0.089  0.060

## Diagnostic score

A single score is used only as a compact diagnostic, not as a definitive model-selection criterion.

We want:

- high similarity for paraphrases;
- a clear decrease for contradictions;
- a clear decrease for lexical-overlap cases;
- low similarity for unrelated pairs;
- related pairs to remain more similar than unrelated pairs.

We therefore compute:

$$
D = (P-C) + (P-L) + (R-U),
$$

where $P,C,L,R,U$ are the mean cosine similarities for paraphrase, contradiction, lexical overlap, related and unrelated pairs respectively.

A larger value indicates stronger separation under this particular benchmark. The individual category means should always be inspected alongside $D$.

In [45]:
means = (
    results_df
    .groupby(["model", "category"])["cosine"]
    .mean()
    .unstack()
)

diagnostic = pd.DataFrame(index=means.index)
diagnostic["paraphrase"] = means["paraphrase"]
diagnostic["contradiction"] = means["contradiction"]
diagnostic["lexical_overlap"] = means["lexical_overlap"]
diagnostic["related"] = means["related"]
diagnostic["unrelated"] = means["unrelated"]

diagnostic["P-C"] = diagnostic["paraphrase"] - diagnostic["contradiction"]
diagnostic["P-L"] = diagnostic["paraphrase"] - diagnostic["lexical_overlap"]
diagnostic["R-U"] = diagnostic["related"] - diagnostic["unrelated"]
diagnostic["diagnostic_score"] = (
    diagnostic["P-C"]
    + diagnostic["P-L"]
    + diagnostic["R-U"]
)

display(diagnostic.round(3).sort_values("diagnostic_score", ascending=False))

,paraphrase,contradiction,lexical_overlap,related,unrelated,P-C,P-L,R-U,diagnostic_score
model,,,,,,,,,
MPNet,0.787,0.606,0.518,0.418,0.025,0.181,0.270,0.392,0.843
MiniLM,0.799,0.769,0.599,0.520,0.009,0.030,0.200,0.511,0.742
BGE-base,0.896,0.795,0.704,0.731,0.399,0.101,0.192,0.332,0.624
BGE-small,0.917,0.814,0.737,0.764,0.442,0.103,0.180,0.321,0.604


## 7. Focus on negation

This is the specific failure mode that motivated the model comparison.

For example:

> `I am studying statistics at EPFL.`

versus

> `I do not study statistics at EPFL.`

A high similarity is not necessarily a failure of sentence embeddings in
general, but it is undesirable if the embedding is intended to act as a
fine-grained content-preservation measure.

We therefore inspect all contradiction pairs separately.

In [46]:
focus = results_df[
    results_df["category"].isin(["contradiction", "lexical_overlap"])
][["model", "category", "cosine", "sentence_1", "sentence_2"]]

display(focus.sort_values(["category", "model", "cosine"]))

,model,category,cosine,sentence_1,sentence_2
57,BGE-base,contradiction,0.746460,I am studying statistics at EPFL.,I do not study statistics at EPFL.
60,BGE-base,contradiction,0.790257,It rained the whole day.,It did not rain during the day.
59,BGE-base,contradiction,0.798639,She works in finance.,She does not work in finance.
58,BGE-base,contradiction,0.845838,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
39,BGE-small,contradiction,0.754042,I am studying statistics at EPFL.,I do not study statistics at EPFL.
42,BGE-small,contradiction,0.810097,It rained the whole day.,It did not rain during the day.
40,BGE-small,contradiction,0.841039,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
41,BGE-small,contradiction,0.850797,She works in finance.,She does not work in finance.
24,MPNet,contradiction,0.479438,It rained the whole day.,It did not rain during the day.
21,MPNet,contradiction,0.606829,I am studying statistics at EPFL.,I do not study statistics at EPFL.


## 8. Select the model

After running the benchmark, choose the model based on the **pattern of
results**, not only on the diagnostic score.

In particular, check:

1. Are paraphrases consistently close?
2. Are explicit contradictions clearly separated from paraphrases?
3. Are unrelated sentences low?
4. Is the behaviour reasonably consistent across different examples?
5. Is the model computationally practical for the later VoxCeleb-scale
   experiment?

Once a model is selected, record its name below and use exactly the same
model in the ASR notebook.